# Exercise 8: Put all the concepts in Exercise 7 together

Skills:
* Apply all the concepts covered in Exercise 7 for a research question. Know when to use what concept.

References: 
* Exercise 7


### To Do

Narrow down the list of rail routes in CA to 3 groups. Use the SHN network to determine how much of the rail route runs near the SHN. We care only about rail routes that run entirely in CA (use stops to figure this out).

**Near** the interstate, US highway, or state highway is defined by being within a quarter mile. For this exercise, the distinction between interstate, US highway, and state highway is not important; treat any road that shows up in the dataset as "the SHN".

There are theoretically 3 groupings: 
* rail routes that are never within 0.25 miles of the SHN
* rail routes with > 0 but less than half of its length near the SHN 
* rail routes with at least half of its length near the SHN

Provide a table and a chart showing how many rail routes fall into each of the 3 groups by district.

Use a Markdown cell at the end to connect which geospatial concept was applied to which step of the process. The concepts that should be used are `projecting CRS`, `buffering`, `dissolve`, `clipping`, `spatial join`, `overlay`. 

In [ ]:
import geopandas as gpd
import intake
import pandas as pd
from shared_utils import geography_utils
catalog = intake.open_catalog(
    "../_shared_utils/shared_utils/shared_data_catalog.yml")

In [ ]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Import data
districts = catalog.caltrans_districts.read()
highways = catalog.state_highway_network.read()

rail_group = ['0', '1', '2']
routes = catalog.ca_transit_routes.read()
rail_routes = routes[routes.route_type.isin(rail_group)
                    ].reset_index(drop=True)
stops = catalog.ca_transit_stops.read()
rail_stops= stops[stops.route_type.isin(rail_group)
                  ].reset_index(drop=True)

### Look at data

In [ ]:
rail_stops.shape

In [ ]:
rail_stops.sample()

In [ ]:
rail_routes.drop(columns = ['geometry']).sample()

In [ ]:
rail_routes.shape

In [ ]:
highways.drop(columns = ['geometry']).head(6)

In [ ]:
highways.head().plot(column = 'Route')

In [ ]:
highways.head().plot(column = 'Direction')

#### Find only stops in California

In [ ]:
def california():
    # Dissolve district to be California only
    gdf = districts.dissolve()[['geometry','Shape__Area','Shape__Length']]
    return gdf

In [ ]:
ca = california()

In [ ]:
ca.crs

In [ ]:
rail_stops.crs

In [ ]:
# Find only stops in California
rail_ca_stops = rail_stops.clip(ca)

In [ ]:
f"{len(rail_stops)-len(rail_ca_stops)} stops are not in CA"

In [ ]:
rail_ca_stops.shape

In [ ]:
rail_ca_stops.plot()

#### Find only routes in California
* need to dissolve at some point?

In [ ]:
def buffer_geo(gdf):
    gdf = gdf.to_crs("EPSG:2229")
    
    # Change to 1/4 of a mile
    gdf = gdf.assign(
    geometry_buffered = gdf.geometry.buffer(50)
    )
    
    return gdf

In [ ]:
rail_routes_buffered = buffer_geo(rail_routes)

In [ ]:
rail_routes_buffered = rail_routes_buffered.set_geometry('geometry_buffered')

In [ ]:
# Why is it so much lighter
# Try explore 
rail_routes_buffered.plot()

In [ ]:
rail_routes.crs

In [ ]:
rail_routes.plot()

In [ ]:
# Find only stops in California by clipping w/o regard to stops
rail_ca_routes_clipped = rail_routes_buffered.clip(ca.to_crs(rail_routes_buffered.crs))

In [ ]:
rail_ca_routes_clipped.route_id.nunique()

In [ ]:
rail_ca_routes_clipped.plot()

In [ ]:
# Merge with routes?
rail_ca_routes_m = pd.merge(
    rail_ca_routes_clipped,
    rail_ca_stops[["agency","route_id","route_type"]],
    how = "inner",
    on = ["agency","route_id","route_type"],
    indicator = True
)

In [ ]:
rail_ca_routes_m._merge.value_counts()

In [ ]:
type(rail_ca_routes_m)

In [ ]:
rail_ca_routes_m.route_id.nunique()

In [ ]:
rail_ca_routes_m = rail_ca_routes_m[['agency','route_id','route_type','route_name','geometry']]

In [ ]:
rail_ca_roues_sjoin = gpd.sjoin(
    rail_routes_buffered,
    rail_ca_stops[['geometry']].to_crs("EPSG:2229"),
    how = "inner",
    predicate = "intersects"
)


In [ ]:
rail_ca_roues_sjoin.shape

In [ ]:
rail_ca_roues_sjoin.route_id.nunique()

In [ ]:
rail_ca_roues_sjoin.plot()

In [ ]:
# Check that route IDS are the same for each method
clipped_routes = set(rail_ca_routes_clipped.route_id.unique().tolist())
merged_routes = set(rail_ca_routes_m.route_id.unique().tolist())
sjoin_routes = set(rail_ca_roues_sjoin.route_id.unique().tolist())

In [ ]:
clipped_routes - merged_routes - sjoin_routes

In [ ]:
# Why do the lengths differ?
len(rail_ca_roues_sjoin),len(rail_ca_routes_m), len(rail_ca_routes_clipped)

#### Highway intersection

In [ ]:
highways_buffered = buffer_geo(highways)

In [ ]:
highways_buffered.crs

In [ ]:
highways_buffered = highways_buffered.set_geometry('geometry_buffered')

In [ ]:
highways_dissolved = highways_buffered.drop(columns = ['geometry','Direction']).dissolve(['Route','County','District', 'RouteType']).reset_index()

In [ ]:
highways_dissolved.drop(columns = ['geometry_buffered']).sample()

In [ ]:
highways_dissolved.shape

In [ ]:
highways_dissolved.Route.nunique()

In [ ]:
highways_dissolved.plot(figsize=(4, 4), column="Route")

#### Overlay
* need to dissolve at some point.

In [ ]:
highways_dissolved['original_geometry_hwy'] = highways_dissolved.geometry_buffered

In [ ]:
rail_ca_roues_sjoin['original_geometry_routes'] = rail_ca_roues_sjoin.geometry_buffered

In [ ]:
highways_dissolved = highways_dissolved.to_crs(rail_ca_roues_sjoin.crs)

In [ ]:
highways_dissolved.crs

In [ ]:
overlay1 = gpd.overlay(
        highways_dissolved, rail_ca_roues_sjoin, how="intersection", keep_geom_type=False
    )


In [ ]:
overlay1 = overlay1.assign(
    original_routes_length = overlay1.original_geometry_routes.length,
    new_length = overlay1.geometry.length)

In [ ]:
# 'original_geometry_routes','geometry','original_geometry_hwy'
overlay_subset = ['Route','County', 'District', 'original_routes_length','new_length']

In [ ]:
overlay2 = overlay1[overlay_subset]

#### Categorize
rail routes that are never within 0.25 miles of the SHN

rail routes with > 0 but less than half of its length near the SHN

rail routes with at least half of its length near the SHN

In [ ]:
overlay2 = overlay2.assign(
    difference_in_ft = (overlay2.original_routes_length - overlay2.new_length),
    percentage = overlay2.new_length/overlay2.original_routes_length * 100)

In [ ]:
overlay2.sample(5)

In [ ]:
mile = 5_280

In [ ]:
0.25*mile

In [ ]:
def categorize_routes_shn(row):
    if row.new_length < 0.25*mile:
        return "never within 0.25 miles of the SHN"
    elif 0 < row.percentage < 50:
        return "less than half of its length near the SHN"
    elif 50 < row.percentage:
        return "at least half of its length near the SHN"
    else:
        return "N/A"

In [ ]:
overlay2["shn_category"] = overlay2.apply(categorize_routes_shn, axis=1)

In [ ]:
overlay2.sample(5)

In [ ]:
overlay_groupby = overlay2.groupby(['District', 'Route','shn_category']).agg({'percentage':'max'}).reset_index()

In [ ]:
overlay_groupby.shape

In [ ]:
overlay2.groupby(['District','shn_category',]).agg({'Route':'nunique'})

In [ ]:
overlay_groupby.shn_category.value_counts()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
f, ax = plt.subplots()

In [ ]:
highways.to_crs(rail_routes.crs).plot(ax=ax)
rail_routes.plot(ax=ax)

In [ ]:
highways.plot()